# 4.7 Grad-CAM 模型視覺化

這份 Notebook 使用小型 CNN 與方向圖形資料集示範 Grad-CAM，觀察模型做分類時關注的影像區域。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

tf.keras.utils.set_random_seed(42)


## 2. 建立方向圖形資料集

這份資料的有效區域很直觀，因此適合檢查 Grad-CAM heatmap 是否落在線條附近。


In [ ]:
CLASS_NAMES = np.array(['vertical', 'horizontal', 'diagonal'])
IMAGE_SIZE = 32

def draw_shape(label, rng, image_size=IMAGE_SIZE, noise=0.12, jitter=3, thickness=2):
    image = rng.normal(loc=0.10, scale=noise, size=(image_size, image_size, 3)).astype('float32')
    color = np.array([0.90, 0.92, 0.88], dtype='float32') + rng.normal(0, 0.03, size=3).astype('float32')
    center = image_size // 2 + int(rng.integers(-jitter, jitter + 1))
    shift = int(rng.integers(-jitter, jitter + 1))

    if label == 0:
        col_start = max(0, center - thickness)
        col_end = min(image_size, center + thickness + 1)
        image[:, col_start:col_end, :] = color
    elif label == 1:
        row_start = max(0, center - thickness)
        row_end = min(image_size, center + thickness + 1)
        image[row_start:row_end, :, :] = color
    else:
        for row in range(image_size):
            col = row + shift
            if 0 <= col < image_size:
                col_start = max(0, col - thickness)
                col_end = min(image_size, col + thickness + 1)
                image[row, col_start:col_end, :] = color

    image += rng.normal(0, noise * 0.6, size=image.shape).astype('float32')
    return np.clip(image, 0.0, 1.0).astype('float32')

def make_shape_dataset(samples_per_class, noise=0.12, jitter=3, seed=42):
    rng = np.random.default_rng(seed)
    images, labels = [], []
    for label in range(len(CLASS_NAMES)):
        for _ in range(samples_per_class):
            images.append(draw_shape(label, rng, noise=noise, jitter=jitter))
            labels.append(label)
    images = np.stack(images).astype('float32')
    labels = np.array(labels, dtype='int64')
    index = rng.permutation(len(labels))
    return images[index], labels[index]

x_train, y_train = make_shape_dataset(samples_per_class=160, noise=0.12, jitter=4, seed=11)
x_test, y_test = make_shape_dataset(samples_per_class=60, noise=0.14, jitter=5, seed=12)
print('train:', x_train.shape, 'test:', x_test.shape)


## 3. 建立含 `last_conv` 的 CNN

Grad-CAM 需要指定一個卷積層。本範例將最後一個卷積層命名為 `last_conv`，方便後續取得 feature maps。


In [ ]:
inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = tf.keras.layers.Conv2D(24, 3, activation='relu', padding='same')(inputs)
x = tf.keras.layers.MaxPooling2D()(x)
x = tf.keras.layers.Conv2D(48, 3, activation='relu', padding='same')(x)
x = tf.keras.layers.MaxPooling2D()(x)
x = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same', name='last_conv')(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


## 4. 訓練模型


In [ ]:
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    verbose=0,
)
pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 4), title='Training Curve')
plt.ylim(0, 1.05)
plt.show()


## 5. 評估並選一張測試圖片


In [ ]:
test_result = model.evaluate(x_test, y_test, verbose=0, return_dict=True)
print(test_result)

sample_index = 0
sample = x_test[sample_index:sample_index + 1]
prob = model.predict(sample, verbose=0)[0]
pred_label = int(np.argmax(prob))
true_label = int(y_test[sample_index])
print('true:', CLASS_NAMES[true_label], 'pred:', CLASS_NAMES[pred_label], 'confidence:', float(prob[pred_label]))

plt.imshow(sample[0])
plt.title(f'true={CLASS_NAMES[true_label]}, pred={CLASS_NAMES[pred_label]}')
plt.axis('off')
plt.show()


## 6. 建立 Grad-CAM 函式


In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.Model(
        model.inputs,
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_score = predictions[:, pred_index]

    grads = tape.gradient(class_score, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

heatmap = make_gradcam_heatmap(sample, model, 'last_conv', pred_label)
print('heatmap shape:', heatmap.shape)


## 7. 將 heatmap 疊回原圖


In [ ]:
def overlay_heatmap(image, heatmap, alpha=0.45):
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], image.shape[:2]).numpy().squeeze()
    cmap = plt.get_cmap('jet')
    colored_heatmap = cmap(heatmap_resized)[..., :3]
    overlay = np.clip((1 - alpha) * image + alpha * colored_heatmap, 0, 1)
    return heatmap_resized, overlay

heatmap_resized, overlay = overlay_heatmap(sample[0], heatmap)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(sample[0])
axes[0].set_title('Original')
axes[1].imshow(heatmap_resized, cmap='jet')
axes[1].set_title('Grad-CAM')
axes[2].imshow(overlay)
axes[2].set_title('Overlay')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


## 8. 批次查看多張圖片


In [ ]:
indices = [0, 20, 40, 80, 120, 160]
plt.figure(figsize=(10, 6))
for plot_i, idx in enumerate(indices, 1):
    img = x_test[idx:idx + 1]
    pred = int(np.argmax(model.predict(img, verbose=0)[0]))
    hm = make_gradcam_heatmap(img, model, 'last_conv', pred)
    _, ov = overlay_heatmap(img[0], hm)
    ax = plt.subplot(2, 3, plot_i)
    ax.imshow(ov)
    ax.set_title(f'true={CLASS_NAMES[y_test[idx]]}\npred={CLASS_NAMES[pred]}')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 9. 如何套用自己的模型

套用到自己的 CNN 時，先用 `model.summary()` 找到後段卷積層名稱，替換 `last_conv_layer_name`。輸入圖片的 resize、normalization 與 preprocessing 必須和模型訓練時一致。若使用 MobileNetV2、ResNet 或 EfficientNet，也要使用對應的 preprocessing function。


## 10. 小結

Grad-CAM 可以輔助檢查 CNN 是否看向合理區域。它不能取代 accuracy、confusion matrix 或錯誤案例分析，但能讓影像模型的診斷更具體。
